# 00 - Uvod i metodologija: Financijske mreze na trzistu kapitala

Autor: Silvija Vlah Jeric, EFZG | Trziste: ZSE/CROBEX 2004-2026

> Napomena: materijal iz tekuceg istrazivackog projekta. Rezultati nisu objavljeni. Prikazana je iskljucivo metodologija.


In [ ]:
# Postavljanje okolisa (automatski detektira Google Colab)
import sys
if "google.colab" in sys.modules:
    import subprocess, os
    subprocess.run(["git", "clone",
                    "https://github.com/svlah-sketch/FinNet-teaching.git",
                    "/content/FinNet-teaching"], check=True)
    os.chdir("/content/FinNet-teaching/notebooks")
    print("Colab: repozitorij kloniran.")
else:
    print("Lokalno pokretanje.")

## 1. Zasto mrezna analiza financijskih trzista?

Klasicni modeli (Markowitz, CAPM) sazimaju odnose medu dionicama u jednu korelacijsku matricu. Mrezna analiza ide dalje: **struktura** te matrice nosi vlastitu informaciju o stanju trzista.

| Pitanje | Mjera |
|---------|-------|
| Koliko je trziste integrirano? | Velicina najvece komponente (LCC) |
| Koliko brzo se sire sokovi? | Prosjecna duljina puta |
| Postoje li izolirane grupe? | Broj zajednica, modularnost |
| Koje su dionice hub sistemskog rizika? | Eigenvector centralnost |
| Koliko je rizik koncentriran? | Apsorcijski omjer |

**Literatura:** Mantegna (1999); Billio et al. (2012) - vidi Sekciju 7.


## 2. Metodoloski pipeline: od prinosa do mreze

**Ovo je kljucna sekcija.**

### Korak 1: Dnevni log-prinosi

$$r_{it} = \log\left(\frac{P_{it}}{P_{i,t-1}}\right)$$

### Korak 2: EW-LOO trzisni faktor

Za svaku dionicu $i$ posebno - jednako-ponderirani prosjek svih **ostalih** dionica:

$$m_{it} = \frac{1}{N-1} \sum_{j \neq i} r_{jt}$$

Zasto ne standardni indeks? Kada bi se koristio puni indeks, regresija $r_{it} = \alpha_i + \beta_i \cdot I_t + \varepsilon_{it}$ bila bi cirkularna: $I_t$ je ponderirana suma svih $r_{jt}$ ukljucujuci i $r_{it}$ sam. To napuhuje betu i pristrano iskrivljuje rezidualne korelacije (Roll, 1977; Cohen, 2025). EW-LOO eliminira taj problem konstrukcijom.

> **Napomena:** U istrazivanju su paralelno koristena oba faktora - EW-LOO i puni CROBEX indeks (oznake `/loo` i `/idx` u tablicama rezultata). Usporedba pokazuje koliko je izbor trzisnog faktora utjece na nalaize - a to je samo po sebi informativno.

### Korak 3: OLS regresija - reziduali (idiosinkratska komponenta)

$$r_{it} = \alpha_i + \beta_i \cdot m_{it} + \varepsilon_{it}$$

Rezidual $\varepsilon_{it}$ je ono sto dionica radi *iznad ili ispod* zajednickog trzisnog kretanja.

### Korak 4: Parcijalna korelacija para $(i, j)$

$$\rho_{ij}^{\text{parc}} = \text{corr}(\varepsilon_{it},\, \varepsilon_{jt})$$

> **Kljucno:** **parcijalna korelacija nije sirova korelacija.**  
> Sirova: $\text{corr}(r_{it}, r_{jt})$ - sadrzi i zajednicki trzisni signal.  
> Parcijalna: $\text{corr}(\varepsilon_{it}, \varepsilon_{jt})$ - iskljucivo idiosinkratska veza.

### Korak 5: P-prag filtriranje - PG i NG podmreze

Za svaki par testiramo $H_0: \rho_{ij}^{\text{parc}} = 0$. Brid postoji ako je $p_{ij} < \alpha$.

- **PG (Positive Graph):** $\rho_{ij}^{\text{parc}} > 0$ - idiosinkratski krecu se **zajedno**
- **NG (Negative Graph):** $\rho_{ij}^{\text{parc}} < 0$ - idiosinkratski krecu se **suprotno**

> **Negativna parcijalna korelacija NE znaci da dionice padaju zajedno.**  
> Zajednicko padanje = zajednicki trzisni faktor = *pozitivna* sirova korelacija.  
> Negativna parcijalna znaci: kada dionica $i$ nadmasuje trziste (poz. rezidual), dionica $j$ ga podmasuje (neg. rezidual) - i obrnuto.  
> To su **relativno suprotna kretanja u idiosinkratskim komponentama** - nakon uklanjanja zajednickog faktora.


## 3. Podaci: CROBEX i Zagrebacka burza

| Karakteristika | Detalj |
|---|---|
| Tip trzista | Granicno (frontier) |
| Dionice | 78 (sve ikad bile u CROBEX, 2004-2026) |
| Frekvencija | Dnevno, CT model trgovanja |
| Valuta | HRK do 31.12.2022; EUR od 1.1.2023 (fiksni tecaj 7.53450) |
| Prozora W90 | ca. 59 |
| Rev. prozora | 43 (rev. 10-52) |

**Krizni periodi:**
- GFC: 1.10.2007 - 30.9.2009
- EU dug: 1.4.2011 - 30.9.2012
- COVID: 20.2.2020 - 31.3.2021


## 4. Deset mreznih mjera

| ID | Naziv | Opis | Mreza | Prag |
|----|-------|------|-------|------|
| M1 | LCC frakcija | Udio dionica u najvecoj komponenti | NG | 0.05 |
| M2 | Avg path length | Prosj. duljina najkraceg puta | NG | 0.05 |
| M3 | Mean degree | Prosj. broj bridova po cvoru | NG | 0.05 |
| M4 | # zajednica | Broj Louvain zajednica | NG | 0.001* |
| M5 | Modularnost | Snaga klaster strukture | NG | 0.001* |
| M6 | Apsorcijski omjer | Sistemski rizik (spektralni) | Svi | - |
| M7 | Mean closeness | Prosj. centralnost blizine | NG | 0.10 |
| M8 | Mean eigenvector | Prosj. eigenvector centralnost | NG | 0.001* |
| M9 | Max eigenvector | Maks. eigenvector centralnost | PG | 0.10 |
| M10 | Avg parc. kor. | Prosj. parcijalna korelacija svih parova | - | - |

*M4, M5, M8 dostupne samo kada je NG-0.001 neprazna (ca. 30-40% prozora).

**M10** je jedina mjera direktno iz matrice parcijalnih korelacija (ne iz grafa).


## 5. Ilustrativni primjer: sirove vs. parcijalne korelacije

**Sinteticki podaci** (ne stvarne cijene). 12 dionica, 3 sektora, 90 dana.

Dva scenarija:
- **Mirno trziste**: umjeren zajednicki faktor, slabija sektorska divergencija
- **Krizni scenarij**: jak zajednicki pad + sektorska divergencija
  (sektor A "osjetljiv" - financije, pada vise od prosjeka;
   sektor C "otporan" - komunalne usluge, pada manje od prosjeka)

Prikaz: sirova vs. parcijalna matrica korelacija + PG/NG mreze.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy import stats
from numpy.linalg import lstsq
from matplotlib.patches import Patch

np.random.seed(42)
N_STOCKS = 12
N_DAYS   = 90
LABELS   = [f"A{i}" for i in range(1,5)] + [f"B{i}" for i in range(1,5)] + [f"C{i}" for i in range(1,5)]
SECTORS  = [0]*4 + [1]*4 + [2]*4
COLORS   = {"A": "#d62728", "B": "#1f77b4", "C": "#2ca02c"}
ALPHA    = 0.10

def simulate(market_std, idio_std, sector_extra=None):
    market = np.random.randn(N_DAYS) * market_std
    sf     = np.random.randn(N_DAYS, 3) * 0.3
    if sector_extra:
        for s, v in sector_extra.items():
            sf[:, s] += v
    idio = np.random.randn(N_DAYS, N_STOCKS) * idio_std
    R = np.zeros((N_DAYS, N_STOCKS))
    for i in range(N_STOCKS):
        R[:, i] = 0.5*market + 0.4*sf[:, SECTORS[i]] + idio[:, i]
    return R

R_norm = simulate(0.5, 0.8)
R_cris = simulate(1.5, 1.2, sector_extra={
    0: np.random.randn(N_DAYS)*(-1.0),
    2: np.random.randn(N_DAYS)*(0.7)})

def ew_loo(R):
    return (R.sum(axis=1, keepdims=True) - R) / (R.shape[1]-1)

def get_resid(R, M):
    E = np.zeros_like(R)
    for i in range(R.shape[1]):
        X = np.column_stack([np.ones(N_DAYS), M[:,i]])
        coef, *_ = lstsq(X, R[:,i], rcond=None)
        E[:,i] = R[:,i] - X @ coef
    return E

def corr_mat(X):
    k = X.shape[1]
    rho, pval = np.zeros((k,k)), np.ones((k,k))
    for i in range(k):
        for j in range(i+1,k):
            r, p = stats.pearsonr(X[:,i], X[:,j])
            rho[i,j]=rho[j,i]=r; pval[i,j]=pval[j,i]=p
    return rho, pval

E_norm = get_resid(R_norm, ew_loo(R_norm))
E_cris = get_resid(R_cris, ew_loo(R_cris))
rho_rn, p_rn = corr_mat(R_norm)
rho_rc, p_rc = corr_mat(R_cris)
rho_pn, p_pn = corr_mat(E_norm)
rho_pc, p_pc = corr_mat(E_cris)

idx = np.triu_indices(N_STOCKS, k=1)
print("Prosjecna korelacija (gornji trokut):")
print(f"  Sirova:     mirno={rho_rn[idx].mean():+.3f}   kriza={rho_rc[idx].mean():+.3f}")
print(f"  Parcijalna: mirno={rho_pn[idx].mean():+.3f}   kriza={rho_pc[idx].mean():+.3f}")
print()
print("Napomena: sirova korelacija u krizi raste (sve padaju zajedno => POZITIVNA).")
print("Parcijalna moze biti negativna (A pada vise, C manje od prosjeka trzista).")


In [ ]:
# --- 4 korelacijske matrice ---
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
panels = [
    ("Mirno - SIROVA corr(r_it, r_jt)", rho_rn),
    ("Kriza - SIROVA corr(r_it, r_jt)", rho_rc),
    ("Mirno - PARCIJALNA corr(e_it, e_jt)", rho_pn),
    ("Kriza - PARCIJALNA corr(e_it, e_jt)", rho_pc),
]
for ax, (title, rho) in zip(axes.flat, panels):
    im = ax.imshow(rho, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(N_STOCKS)); ax.set_xticklabels(LABELS, rotation=45, fontsize=7)
    ax.set_yticks(range(N_STOCKS)); ax.set_yticklabels(LABELS, fontsize=7)
    for pos in [3.5, 7.5]:
        ax.axhline(pos, color="k", lw=1.5); ax.axvline(pos, color="k", lw=1.5)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(
    "Sirova vs. parcijalna korelacija | Mirno vs. krizno trziste\n"
    "(sinteticki podaci: A=osjetljivi, B=neutralni, C=otporni sektor)", fontsize=11)
plt.tight_layout()
plt.savefig("nb00_korelacije.png", dpi=120, bbox_inches="tight")
plt.show()
print("""
Gornji red (SIROVA):
  Mirno: pretezno pozitivne, blok-struktura po sektorima
  Kriza: sve korelacije rastu - dijagonala i van nje crvena.
         Dionice padaju zajedno => to je POZITIVNA sirova korelacija!

Donji red (PARCIJALNA - reziduali, EW-LOO uklonjen):
  Mirno: slabije, blize nuli
  Kriza: negativne parcijalne (plavo) izmedju A i C.
         A pada VISE od prosjeka (negativan rezidual).
         C pada MANJE (pozitivan rezidual).
         Negativna parcijalna korelacija => brid u NG mrezi.
""")


In [ ]:
# --- PG i NG mreze ---
def build_graphs(rho, pval):
    G_PG, G_NG = nx.Graph(), nx.Graph()
    G_PG.add_nodes_from(LABELS); G_NG.add_nodes_from(LABELS)
    for i in range(N_STOCKS):
        for j in range(i+1, N_STOCKS):
            if pval[i,j] < ALPHA:
                G = G_PG if rho[i,j] > 0 else G_NG
                G.add_edge(LABELS[i], LABELS[j], weight=abs(rho[i,j]))
    return G_PG, G_NG

def draw_g(ax, G, title):
    nc = [COLORS[l[0]] for l in LABELS]
    pos = nx.spring_layout(G, seed=42) if G.number_of_edges() else nx.circular_layout(G)
    w = [G[u][v]["weight"]*4 for u,v in G.edges()] if G.number_of_edges() else []
    nx.draw_networkx(G, pos=pos, ax=ax, node_color=nc, node_size=500,
                     font_size=8, font_color="white", edge_color="gray",
                     width=w if w else 1, with_labels=True)
    lcc = max((len(c) for c in nx.connected_components(G)), default=1) if G.number_of_edges() else 1
    ax.set_title(f"{title}\n{G.number_of_edges()} bridova | LCC={lcc}/{N_STOCKS}", fontsize=9)
    ax.axis("off")

PG_n, NG_n = build_graphs(rho_pn, p_pn)
PG_c, NG_c = build_graphs(rho_pc, p_pc)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
draw_g(axes[0,0], PG_n, f"PG mirno (alpha={ALPHA})")
draw_g(axes[0,1], PG_c, f"PG kriza (alpha={ALPHA})")
draw_g(axes[1,0], NG_n, f"NG mirno (alpha={ALPHA})")
draw_g(axes[1,1], NG_c, f"NG kriza (alpha={ALPHA})")
leg = [Patch(color=c, label=f"Sektor {s}") for s,c in COLORS.items()]
fig.legend(handles=leg, loc="lower center", ncol=3, fontsize=10, frameon=False)
fig.suptitle(f"PG i NG mreze od PARCIJALNIH korelacija\n"
             "Mirno vs. krizno trziste - sinteticki podaci", fontsize=12)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("nb00_mreze.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"""
NG mirno:  {NG_n.number_of_edges()} bridova
NG kriza:  {NG_c.number_of_edges()} bridova

ZAKLJUCAK: NG se puni u krizi zbog sektorske divergencije.
To nisu dionice koje se krecu u suprotnim apsolutnim smjerovima.
To su dionice ciji REZIDUALI imaju suprotne predznake:
  A: pada vise nego sto trzisni faktor predvidja (neg. rezidual)
  C: pada manje (poz. rezidual)
=> negativna PARCIJALNA korelacija => brid u NG
""")


## 6. Sazetak: sirove vs. parcijalne korelacije

| | Sirova | Parcijalna |
|--|---|---|
| **Formula** | corr(r_it, r_jt) | corr(e_it, e_jt) |
| **Sto mjeri** | Zajednicki faktor + idiosinkratika | Samo idiosinkratika |
| **U krizi** | Raste (sve padaju zajedno) | Moze biti negativna (divergentni sektori) |
| **NG u krizi** | Rijetko pun | Puni se - hvata relativnu divergenciju |

**NG ne mjeri "dionice padaju zajedno"** - to bi bila pozitivna sirova korelacija.  
NG mjeri: dionica A pada **vise** od trzista, dionica C **manje** - njihovi reziduali su negativno korelirani.

---

## 7. Literatura

### Mrezna analiza financijskih trzista
- Mantegna, R.N. (1999). Hierarchical structure in financial markets. *European Physical Journal B*, 11. https://doi.org/10.1007/s100510050929
- Onnela, J.-P. et al. (2003). Dynamics of market correlations. *Physical Review E*, 68. https://doi.org/10.1103/PhysRevE.68.056110
- Billio, M. et al. (2012). Econometric measures of connectedness and systemic risk. *Journal of Financial Economics*, 104(3). https://doi.org/10.1016/j.jfineco.2011.12.010
- Kenett, D.Y. et al. (2010). Dominating clasp of the financial sector revealed by partial correlation analysis. *PLoS ONE*, 5(12), e15032. https://doi.org/10.1371/journal.pone.0015032 *(open access)*
- Xu, R. et al. (2017). Topological characteristics of the Hong Kong stock market: a test-based P-threshold approach. *Scientific Reports*, 7, 41379. https://doi.org/10.1038/srep41379 *(open access)*

### Mrezne mjere i sistemski rizik
- Newman, M.E.J. (2010). *Networks: An Introduction*. Oxford University Press.
- Blondel, V.D. et al. (2008). Fast unfolding of communities in large networks. *Journal of Statistical Mechanics*, P10008. https://doi.org/10.1088/1742-5468/2008/10/P10008
- Kritzman, M. et al. (2011). Principal components as a measure of systemic risk. *Journal of Portfolio Management*, 37(4). https://doi.org/10.3905/jpm.2011.37.4.112

### EW-LOO trzisni faktor i cirkularnost
- Roll, R. (1977). A critique of the asset pricing theory's tests. *Journal of Financial Economics*, 4(2), 129-176. https://doi.org/10.1016/0304-405X(77)90009-5 *(EFZG knjiznica)*
- Cohen, N. (2025). Rethinking beta: a causal take on CAPM. arXiv:2509.05760. https://arxiv.org/abs/2509.05760 *(preprint, open access)*

### Prognoziranje i statisticki testovi
- Bollerslev, T. (1986). Generalized autoregressive conditional heteroskedasticity. *Journal of Econometrics*, 31(3). https://doi.org/10.1016/0304-4076(86)90063-1
- Diebold, F.X. & Mariano, R.S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3). https://doi.org/10.1080/07350015.1995.10524599
- Harvey, D., Leybourne, S. & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting*, 13(2). https://doi.org/10.1016/S0169-2070(96)00719-4

### Granicna i manja trzista
- Bekaert, G. & Harvey, C.R. (2002). Research in emerging markets finance. *Emerging Markets Review*, 3(4). https://doi.org/10.1016/S1566-0141(02)00045-9
